<a href="https://colab.research.google.com/github/bharghavbolla/641-HW2-Bharghav/blob/main/Starting_Point_for_Project_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Skeleton for the fourth programming project

You will modify a sequential version of the python code that solves the maximum independent set problem introducing MPI4PY calls.

As a first step, look at this code that was provided in class to solve the partition problem using MPI4PY:

https://github.com/trefftzc/partition_COLAB_notebooks/blob/main/partition_mpi4py.ipynb

Notice that you need to add the following code:

1. At the very beginning of the code:
  from mpi4py import MPI
2. At the beginning of the main method  
  comm = MPI.COMM_WORLD
  rank = comm.Get_rank()
  number_nodes = comm.Get_size()
3. The coordinator node, with rank 0, reads the size of the problem and the adjacency matrix.
4. The coordinator node should broadcast the size of the problem and the adjacency matrix to all other nodes.
5. Every node calculates which portion of the values in the main loop it should work on.
6. Every node works on a different portion of the main loop
7. Perform a reduction to find the largest independent set. The results should be placed on node 0, the coordinator. Node 0 will print the result.

Let's start with several test files:


In [ ]:
%%writefile k4.txt
4
0 1 1 1
1 0 1 1
1 1 0 1
1 1 1 0

Overwriting k4.txt


In [ ]:
%%writefile no_edges_4.txt
4
0 0 0 0
0 0 0 0
0 0 0 0
0 0 0 0

Overwriting no_edges_4.txt


In [ ]:
%%writefile k16.txt
16
0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0

Overwriting k16.txt


Now the original sequential python code. It is probably a good idea to keep it without modifications to compare the results.

In [ ]:
%%writefile original_python.py
import sys
import time
import numpy as np

def read_adjacency_matrix(file_name):
  file_object = open(file_name, "r")
  # Input the number of rows and columns
  size = int(file_object.readline())
  rows = size
  cols = size
  # Initialize an empty matrix
  matrix = []

  # Input the matrix elements
  for i in range(rows):
    row = list(map(int, file_object.readline().split()))
    matrix.append(row)

  return matrix,size

# Convert an integer into a set of nodes
def convert_from_int_to_set(integer,size):
  set_of_nodes = []
  mask = 1
  for i in range(size):
    if ((mask & integer) != 0):
      set_of_nodes.append(i)
    mask = mask * 2
  return set_of_nodes

# Find the maximum independent set
def find_max_ind_set(adj_mat_numpy,size):
  max_independent_set_size = 0
  max_independent_set = []

  size_of_power_set = 1
  for i in range(size):
    size_of_power_set *= 2
  # print("The power set has ",size_of_power_set," elements")
  array_with_sizes = np.zeros(size_of_power_set)
  for i in range(size_of_power_set):
    this_set = convert_from_int_to_set(i,size)
    is_independent = True
    for n1 in this_set:
      for n2 in this_set:
        if (adj_mat_numpy[n1][n2] == 1):
          is_independent = False
    if (is_independent):
      array_with_sizes[i] = len(this_set)
    else:
      array_with_sizes[i] = 0


  max_independent_set_size = np.max(array_with_sizes)
  max_independent_set = np.where(array_with_sizes == max_independent_set_size)[0]
  print("The max independent sets are encoded by: ",max_independent_set)
  return max_independent_set_size



if __name__ == "__main__":
# Read the content of the file with the a passed in the command line
# that contain the matrices to be multiplied
  adj_matrix,size = read_adjacency_matrix(sys.argv[1])
  adj_matrix_numpy = np.array(adj_matrix)
  start_time = time.time()
  max_independent_set_size = find_max_ind_set(adj_matrix_numpy,size)
  end_time = time.time()
  elapsed_time = end_time - start_time
  print("Time required to carry out the computation in python: ",elapsed_time)
  print("The size of the maximum independent set is: ",max_independent_set_size)


Overwriting original_python.py


In [ ]:
!python3 original_python.py k4.txt


The max independent sets are encoded by:  [1 2 4 8]
Time required to carry out the computation in python:  0.00036597251892089844
The size of the maximum independent set is:  1.0


Install the mpi4py library:

In [ ]:
!pip install mpi4py

Let's test that mpi4py is working correctly on a very small program.

In [ ]:
%%writefile small_test.py
from mpi4py import MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
number_nodes = comm.Get_size()
print("I am node: ",rank)
print("There are ",number_nodes," copies of this program in this execution.")

Overwriting small_test.py


In [ ]:
!OMPI_ALLOW_RUN_AS_ROOT=1
!mpiexec --allow-run-as-root -n 2 --oversubscribe python small_test.py

I am node:  1
There are  2  copies of this program in this execution.
I am node:  0
There are  2  copies of this program in this execution.


Now a second copy of the original python sequential code.
Change this second version to make sure it works as expected.

In [ ]:
%%writefile with_mpi4py.py
import sys
import time
import numpy as np

def read_adjacency_matrix(file_name):
  file_object = open(file_name, "r")
  # Input the number of rows and columns
  size = int(file_object.readline())
  rows = size
  cols = size
  # Initialize an empty matrix
  matrix = []

  # Input the matrix elements
  for i in range(rows):
    row = list(map(int, file_object.readline().split()))
    matrix.append(row)

  return matrix,size

# Convert an integer into a set of nodes
def convert_from_int_to_set(integer,size):
  set_of_nodes = []
  mask = 1
  for i in range(size):
    if ((mask & integer) != 0):
      set_of_nodes.append(i)
    mask = mask * 2
  return set_of_nodes

# Find the maximum independent set
def find_max_ind_set(adj_mat_numpy,size):
  max_independent_set_size = 0
  max_independent_set = []

  size_of_power_set = 1
  for i in range(size):
    size_of_power_set *= 2
  # print("The power set has ",size_of_power_set," elements")
  array_with_sizes = np.zeros(size_of_power_set)
  for i in range(size_of_power_set):
    this_set = convert_from_int_to_set(i,size)
    is_independent = True
    for n1 in this_set:
      for n2 in this_set:
        if (adj_mat_numpy[n1][n2] == 1):
          is_independent = False
    if (is_independent):
      array_with_sizes[i] = len(this_set)
    else:
      array_with_sizes[i] = 0


  max_independent_set_size = np.max(array_with_sizes)
  max_independent_set = np.where(array_with_sizes == max_independent_set_size)[0]
  print("The max independent sets are encoded by: ",max_independent_set)
  return max_independent_set_size



if __name__ == "__main__":
# Read the content of the file with the a passed in the command line
# that contain the matrices to be multiplied
  adj_matrix,size = read_adjacency_matrix(sys.argv[1])
  adj_matrix_numpy = np.array(adj_matrix)
  start_time = time.time()
  max_independent_set_size = find_max_ind_set(adj_matrix_numpy,size)
  end_time = time.time()
  elapsed_time = end_time - start_time
  print("Time required to carry out the computation in python: ",elapsed_time)
  print("The size of the maximum independent set is: ",max_independent_set_size)

Overwriting with_mpi4py.py


And now the command to execute the code that incorporates MPI4PY functions.

In [ ]:
!OMPI_ALLOW_RUN_AS_ROOT=1
!mpiexec --allow-run-as-root -n 2 --oversubscribe python with_mpi4py.py k4.txt

The max independent sets are encoded by:  [1 2 4 8]
Time required to carry out the computation in python:  0.000514984130859375
The size of the maximum independent set is:  1.0
The max independent sets are encoded by:  [1 2 4 8]
Time required to carry out the computation in python:  0.004678249359130859
The size of the maximum independent set is:  1.0


**CODE**


In [ ]:
!pip install mpi4py


In [ ]:
%%writefile k4.txt
4
0 1 1 1
1 0 1 1
1 1 0 1
1 1 1 0


Overwriting k4.txt


In [ ]:
%%writefile no_edges_4.txt
4
0 0 0 0
0 0 0 0
0 0 0 0
0 0 0 0
%%writefile no_edges_4.txt
4
0 0 0 0
0 0 0 0
0 0 0 0
0 0 0 0
%%writefile k16.txt
16
0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0

Overwriting no_edges_4.txt


In [ ]:
%%writefile with_mpi4py.py
from mpi4py import MPI
import sys
import time
import numpy as np

def read_adjacency_matrix(file_name):
    file_object = open(file_name, "r")
    size = int(file_object.readline())
    rows = size
    cols = size
    matrix = []
    for i in range(rows):
        row = list(map(int, file_object.readline().split()))
        matrix.append(row)
    return matrix, size

def convert_from_int_to_set(integer, size):
    set_of_nodes = []
    mask = 1
    for i in range(size):
        if ((mask & integer) != 0):
            set_of_nodes.append(i)
        mask = mask * 2
    return set_of_nodes

# This function plays the role of solve() in partition_mpi4py.py
# It has NO MPI calls. It just works on [begin, end) for this node.
def max_ind_set_in_range(adj_mat_numpy, size, begin, end):
    best_size = 0
    # Loop over the portion assigned to this node
    for i in range(begin, end):
        this_set = convert_from_int_to_set(i, size)
        is_independent = True

        # Check if this_set is independent
        for n1 in this_set:
            for n2 in this_set:
                if adj_mat_numpy[n1][n2] == 1:
                    is_independent = False
                    break
            if not is_independent:
                break

        # Update best
        if is_independent:
            set_size = len(this_set)
            if set_size > best_size:
                best_size = set_size

    return best_size


if __name__ == "__main__":
    # --- MPI initialization (same pattern as partition_mpi4py.py) ---
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    number_nodes = comm.Get_size()

    # Coordinator reads the graph
    if rank == 0:
        if len(sys.argv) < 2:
            print("Usage: python with_mpi4py.py <input_file>")
            sys.exit(1)
        adj_matrix, size = read_adjacency_matrix(sys.argv[1])
        print("Graph size (number of nodes):", size)
    else:
        adj_matrix = None
        size = None

    # Broadcast size
    size = comm.bcast(size, root=0)

    # Broadcast adjacency matrix using numpy array (same style as partition code)
    if rank == 0:
        adj_matrix_numpy = np.array(adj_matrix, dtype='i')
    else:
        adj_matrix_numpy = np.empty((size, size), dtype='i')

    adj_matrix_numpy = comm.bcast(adj_matrix_numpy, root=0)

    # Total number of subsets in the power set
    nSubsets = 1 << size  # 2**size

    # Block distribution of the work (same as partition_mpi4py.py)
    portionEachNode = nSubsets // number_nodes
    initial = portionEachNode * rank
    if rank != (number_nodes - 1):
        final = initial + portionEachNode
    else:
        # Last node takes any remainder
        final = nSubsets

    # Timing (rank 0 only, like partition example)
    if rank == 0:
        start = time.time()

    # Each node computes the max independent set SIZE on its portion
    max_in_this_node = max_ind_set_in_range(adj_matrix_numpy, size, initial, final)

    # Reduction to find the largest independent set SIZE on node 0
    global_max_size = comm.reduce(max_in_this_node, op=MPI.MAX, root=0)

    if rank == 0:
        end = time.time()
        elapsed = end - start
        print("The size of the maximum independent set is:", global_max_size)
        print("The program took:", elapsed, "seconds.")


Overwriting with_mpi4py.py


In [ ]:
%%writefile original_mpi.py
import sys
import time
import numpy as np
from mpi4py import MPI

def read_adjacency_matrix(file_name):
    file_object = open(file_name, "r")
    size = int(file_object.readline())
    rows = size
    cols = size
    matrix = []
    for i in range(rows):
        row = list(map(int, file_object.readline().split()))
        matrix.append(row)
    return matrix, size

def convert_from_int_to_set(integer, size):
    set_of_nodes = []
    mask = 1
    for i in range(size):
        if ((mask & integer) != 0):
            set_of_nodes.append(i)
        mask = mask * 2
    return set_of_nodes

def is_independent_set(this_set, adj_mat_numpy):
    for n1 in this_set:
        for n2 in this_set:
            if adj_mat_numpy[n1][n2] == 1:
                return False
    return True

def find_max_ind_set_mpi(adj_mat_numpy, size):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    total_ranks = comm.Get_size()

    size_of_power_set = 1 << size  # 2**size

    # Local best for each rank
    local_best_size = 0
    local_best_mask = 0

    # Cyclic distribution of subsets: rank, rank+size, rank+2*size...
    i = rank
    while i < size_of_power_set:
        this_set = convert_from_int_to_set(i, size)
        if is_independent_set(this_set, adj_mat_numpy):
            s = len(this_set)
            if s > local_best_size:
                local_best_size = s
                local_best_mask = i
        i += total_ranks

    # Gather all results at rank 0
    local_result = (local_best_size, local_best_mask)
    all_results = comm.gather(local_result, root=0)

    if rank == 0:
        # Pick maximum by size
        global_best_size, global_best_mask = max(all_results, key=lambda x: x[0])
    else:
        global_best_size = None
        global_best_mask = None

    # Broadcast results to all processes
    global_best_size = comm.bcast(global_best_size, root=0)
    global_best_mask = comm.bcast(global_best_mask, root=0)

    return global_best_size, global_best_mask


if __name__ == "__main__":
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()

    # Rank 0 reads file
    if rank == 0:
        adj_matrix, size = read_adjacency_matrix(sys.argv[1])
        adj_matrix_numpy = np.array(adj_matrix)
    else:
        adj_matrix_numpy = None
        size = None

    # Broadcast matrix + size
    size = comm.bcast(size, root=0)
    adj_matrix_numpy = comm.bcast(adj_matrix_numpy, root=0)

    # Measure time only at rank 0
    if rank == 0:
        start_time = time.time()

    max_independent_set_size, max_mask = find_max_ind_set_mpi(adj_matrix_numpy, size)

    if rank == 0:
        end_time = time.time()
        elapsed_time = end_time - start_time
        max_set = convert_from_int_to_set(max_mask, size)

        print("Time required to carry out computation with mpi4py:", elapsed_time)
        print("The size of the maximum independent set is:", max_independent_set_size)
        print("One MIS solution is encoded by:", max_mask)
        print("MIS vertices (0-based):", max_set)


Writing original_mpi.py


In [ ]:
# Test with complete graph on 4 nodes
!OMPI_ALLOW_RUN_AS_ROOT=1
!mpiexec --allow-run-as-root -n 2 --oversubscribe python with_mpi4py.py k4.txt


Graph size (number of nodes): 4
The size of the maximum independent set is: 1
The program took: 0.0013957023620605469 seconds.


In [ ]:
# Test with graph that has no edges (no_edges_4.txt)
!OMPI_ALLOW_RUN_AS_ROOT=1
!mpiexec --allow-run-as-root -n 4 --oversubscribe python with_mpi4py.py no_edges_4.txt


Graph size (number of nodes): 4
The size of the maximum independent set is: 4
The program took: 0.07614946365356445 seconds.


In [ ]:
print("Sequential result:")
!python3 original_python.py k4.txt

print("\nMPI result (2 processes):")
!OMPI_ALLOW_RUN_AS_ROOT=1
!mpiexec --allow-run-as-root -n 2 --oversubscribe python with_mpi4py.py k4.txt


Sequential result:
The max independent sets are encoded by:  [1 2 4 8]
Time required to carry out the computation in python:  0.0004000663757324219
The size of the maximum independent set is:  1.0

MPI result (2 processes):
Graph size (number of nodes): 4
The size of the maximum independent set is: 1
The program took: 0.0003261566162109375 seconds.
